# Cleanup: 01 — Data prep teardown

**Removes** artifacts created by `01_gdelt_data_prep.ipynb`:

- BigQuery dataset `{BIGQUERY_DATASET}` (US-CENTRAL1) **and all contents** — partitioned GDELT copies, graph tables, procedures, `_sync_metadata`, etc.
- BigQuery dataset `{BIGQUERY_DATASET}_us` (US) **and all contents** — temp / staging tables used for cross-region copies
- Local **`gephi_exports`** folders and **`gdelt_person_cooccurrence*.gexf`** under home, current working directory, and parent of cwd (matching export fallback paths)

**IAM**: BigQuery admin-level ability to delete datasets; local filesystem write access for export cleanup.

**Irreversible** — all data in those datasets is deleted.

**Config:** `{BIGQUERY_DATASET}` / `{BIGQUERY_DATASET}_us` match **`BIGQUERY_DATASET`** from **`config.py`** (loaded by the code cells below).

This notebook runs **`DROP PROPERTY GRAPH IF EXISTS`** first so you can do a full teardown from this notebook alone even if cleanup 02 was skipped.

### Full reset order

1. **`02_gdelt_graph_create_cleanup.ipynb`** — Drop the BigQuery property graph while datasets still exist.
2. **`03_gdelt_incremental_refresh_cleanup.ipynb`** — Remove `_sync_metadata` and staging tables (optional if you immediately run step 4).
3. **`04_gdelt_data_profiling_cleanup.ipynb`** — Remove Dataplex scans and generated descriptions (dataset must still exist).
4. **`01_gdelt_data_prep_cleanup.ipynb`** — Delete `{BIGQUERY_DATASET}`, `{BIGQUERY_DATASET}_us`, and local exports (**always last**).

```mermaid
flowchart LR
  cleanup02[cleanup_02_drop_graph]
  cleanup03[cleanup_03_incremental_artifacts]
  cleanup04[cleanup_04_dataplex_and_descriptions]
  cleanup01[cleanup_01_datasets_and_local]
  cleanup02 --> cleanup01
  cleanup03 --> cleanup01
  cleanup04 --> cleanup01
```

In [ ]:
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
_cfg_dir = _cwd.parent if _cwd.name == "clean_up" else _cwd
if not (_cfg_dir / "config.py").is_file():
    _cfg_dir = _cwd
sys.path.insert(0, str(_cfg_dir))

from config import GCP_PROJECT_ID, BIGQUERY_DATASET, print_config, validate_config

print_config()
if not validate_config():
    print("\n⚠️  Update config.py before proceeding.")
else:
    print("\n✅ Configuration loaded.")

In [ ]:
import shutil
from pathlib import Path

from google.cloud import bigquery
from google.cloud.exceptions import NotFound

client = bigquery.Client(project=GCP_PROJECT_ID)

ds_ref = bigquery.DatasetReference(GCP_PROJECT_ID, BIGQUERY_DATASET)
try:
    client.get_dataset(ds_ref)
except NotFound:
    print(
        f"ℹ️  Dataset `{GCP_PROJECT_ID}.{BIGQUERY_DATASET}` not found — "
        "skipping property graph drop (already removed or wrong config)."
    )
else:
    graph_fqn = f"`{GCP_PROJECT_ID}.{BIGQUERY_DATASET}.GdeltGraph`"
    drop_graph_sql = f"DROP PROPERTY GRAPH IF EXISTS {graph_fqn}"
    print(f"Executing: {drop_graph_sql}")
    try:
        client.query(drop_graph_sql, location="US-CENTRAL1").result()
        print("✅ Property graph dropped (or did not exist).")
    except Exception as e:
        print(f"⚠️  Property graph step: {e}")

us_dataset_id = f"{BIGQUERY_DATASET}_us"
for ds_id in (us_dataset_id, BIGQUERY_DATASET):
    ref = bigquery.DatasetReference(GCP_PROJECT_ID, ds_id)
    client.delete_dataset(ref, delete_contents=True, not_found_ok=True)
    print(f"✅ Dataset deleted (or absent): {ds_id}")

bases = {Path.home(), Path.cwd(), Path.cwd().parent}
for base in bases:
    try:
        if not base.exists():
            continue
        gephi_dir = base / "gephi_exports"
        if gephi_dir.is_dir():
            shutil.rmtree(gephi_dir)
            print(f"✅ Removed local directory {gephi_dir}")
        for gexf in base.glob("gdelt_person_cooccurrence*.gexf"):
            gexf.unlink(missing_ok=True)
            print(f"✅ Removed {gexf}")
    except Exception as e:
        print(f"⚠️  Local cleanup under {base}: {e}")

print("\n✅ Data prep cleanup finished.")

## Optional sanity check

Confirm datasets are gone from the project listing (may be paginated).

In [ ]:
names = {d.dataset_id for d in client.list_datasets(max_results=500)}
for want in (BIGQUERY_DATASET, f"{BIGQUERY_DATASET}_us"):
    print(f"{want}: {'still present' if want in names else 'not listed'}")